In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import category_encoders as ce
from sklearn.metrics import mean_absolute_error, mean_squared_error
from skopt import BayesSearchCV
from xgboost import XGBRegressor
import pickle

In [2]:
df = pd.read_csv('/home/puru/Documents/House Price Prediction/STEP - 3 (Feature Engg)/gurgaon_properties_post_feature_selection.csv')
X = df.drop(columns=['price'])
y = df['price']
y_transformed = np.log1p(y)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), ['super_built_up_area', 'bedRoom', 'bathroom']),
        ('ordinal_enc', OrdinalEncoder(), ['balcony']),
        ('onehot_enc', OneHotEncoder(drop='first', sparse_output=False), ['property_type', 'agePossession', 'servant room', 'furnishing_type', 'luxury_category', 'floor_category']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [5]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42))
])

In [6]:
param_space = {
    'regressor__n_estimators': (50, 500),
    'regressor__learning_rate': (1e-4, 0.5, 'uniform'),
    'regressor__max_depth': (3, 21), 
    'regressor__subsample': (0.6, 1.0), 
    'regressor__colsample_bytree': (0.6, 1.0),
    'regressor__gamma': (0.1, 0.5),
    'regressor__reg_alpha': (0.5, 1.0),
    'regressor__reg_lambda': (0.5, 2.0)
}


In [7]:
optimizer = BayesSearchCV(
    estimator=pipeline,
    search_spaces=param_space,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42)

In [8]:
optimizer.fit(X_train, y_train)

BayesSearchCV(cv=5,
              estimator=Pipeline(steps=[('preprocessor',
                                         ColumnTransformer(remainder='passthrough',
                                                           transformers=[('num_scaler',
                                                                          StandardScaler(),
                                                                          ['super_built_up_area',
                                                                           'bedRoom',
                                                                           'bathroom']),
                                                                         ('ordinal_enc',
                                                                          OrdinalEncoder(),
                                                                          ['balcony']),
                                                                         ('onehot_enc',
                                                                          OneHotEncoder(drop='first',
                                                                                        sparse_output=False),
                                                                          ['property_type',
                                                                           'agePossession',
                                                                           'servant '
                                                                           'room',...
                                                      random_state=42, ...))]),
              random_state=42, scoring='r2',
              search_spaces={'regressor__colsample_bytree': (0.6, 1.0),
                             'regressor__gamma': (0.1, 0.5),
                             'regressor__learning_rate': (0.0001, 0.5,
                                                          'uniform'),
                             'regressor__max_depth': (3, 21),
                             'regressor__n_estimators': (50, 500),
                             'regressor__reg_alpha': (0.5, 1.0),
                             'regressor__reg_lambda': (0.5, 2.0),
                             'regressor__subsample': (0.6, 1.0)})

In [9]:
y_pred_transformed = optimizer.best_estimator_.predict(X_test)
y_pred = np.expm1(y_pred_transformed)

In [10]:
mse = mean_squared_error(np.expm1(y_test), y_pred) 
mae = mean_absolute_error(np.expm1(y_test), y_pred)

print(f"Mean Squared Error on Test Set: {mse:.4f}")
print(f"Mean Absolute Error on Test Set: {mae:.4f}")

Mean Squared Error on Test Set: 0.9004
Mean Absolute Error on Test Set: 0.4814


### `Export the model`

In [11]:
optimizer.best_estimator_

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num_scaler',
                                                  StandardScaler(),
                                                  ['super_built_up_area',
                                                   'bedRoom', 'bathroom']),
                                                 ('ordinal_enc',
                                                  OrdinalEncoder(),
                                                  ['balcony']),
                                                 ('onehot_enc',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['property_type',
                                                   'agePossession',
                                                   'servant room',
                                                   'furnishing_type',
                                                   'luxury_categ...
                              feature_types=None, gamma=0.1, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None,
                              learning_rate=0.16983507603285525, max_bin=None,
                              max_cat_threshold=None, max_cat_to_onehot=None,
                              max_delta_step=None, max_depth=21,
                              max_leaves=None, min_child_weight=None,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=500,
                              n_jobs=None, num_parallel_tree=None,
                              random_state=42, ...))])

In [12]:
with open('best_pipeline.pkl', 'wb') as f:
    pickle.dump(optimizer.best_estimator_, f)

In [13]:
with open('df.pkl', 'wb') as f:
    pickle.dump(X, f)

#### `Prediction`

In [16]:
data = [['flat', 'sector 85', 2, 2, '2', 'New Property', 500, 'No', 'unfurnished', 'Low', 'High Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony','agePossession', 'super_built_up_area', 'servant room','furnishing_type', 'luxury_category', 'floor_category']
one_df = pd.DataFrame(data, columns=columns)
one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,super_built_up_area,servant room,furnishing_type,luxury_category,floor_category
0,flat,sector 85,2,2,2,New Property,500,No,unfurnished,Low,High Floor


In [17]:
np.expm1(optimizer.best_estimator_.predict(one_df))

array([0.4552092], dtype=float32)